# IND-3 : Défauts synthétiques par diffusion pour l'inspection visuelle

Question de recherche : l'ajout de défauts générés par diffusion améliore-t-il la détection de défauts rares, ou ne fait-il que dupliquer les cas déjà présents ?

## Cellule 1 — Imports + constantes

In [ ]:
import os
import random
import shutil
from datetime import datetime
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

from anomalib.data import Mvtec
from anomalib.engine import Engine
from anomalib.models import Patchcore

# Constantes
SEEDS = [42, 123, 2024]
BATCH_SIZE = 32
IMAGE_SIZE = 256
CATEGORIES = ["bottle", "cable", "transistor"]
REGIMES = ["R1_reel_seul", "R2_aug_classique", "R3_synthetique"]
DATA_DIR = "./data"
RESULTS_DIR = "./results"
FIGURES_DIR = "./figures"
SYNTH_DIR = "./synthetiques"

COLONNES_RESULTS = [
    "timestamp",
    "categorie",
    "regime",
    "seed",
    "auroc_image",
    "auroc_pixel",
    "aupro"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print("Device :", DEVICE)
print("Catégories :", CATEGORIES)
print("Régimes :", REGIMES)

## Cellule 2 — Téléchargement et préparation des données

In [ ]:
def creer_datamodule(categorie, batch_size, image_size, data_dir, regime="R1_reel_seul", synth_dir=None):
    """Crée le datamodule MVTec AD pour une catégorie et un régime donnés."""
    if regime == "R3_synthetique":
        injecter_synthetiques_dans_train(categorie, synth_dir, data_dir)

    train_transform = None
    eval_transform = A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

    if regime == "R2_aug_classique":
        train_transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.Resize(image_size, image_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    datamodule = Mvtec(
        root=data_dir,
        category=categorie,
        image_size=(image_size, image_size),
        train_batch_size=batch_size,
        eval_batch_size=batch_size,
        num_workers=0,
        task="SEGMENTATION",
        train_transform=train_transform,
        eval_transform=eval_transform
    )
    datamodule.prepare_data()
    datamodule.setup()
    return datamodule


def afficher_counts_datamodule(datamodule, categorie):
    """Affiche les nombres d'images par split pour une catégorie."""
    n_train = len(datamodule.train_data) if hasattr(datamodule, "train_data") else 0
    n_test = len(datamodule.test_data) if hasattr(datamodule, "test_data") else 0
    n_val = len(datamodule.val_data) if hasattr(datamodule, "val_data") else 0
    print(f"--- Counts pour {categorie} ---")
    print(f"  Train : {n_train}")
    print(f"  Test  : {n_test}")
    print(f"  Val   : {n_val}")


for cat in CATEGORIES:
    dm = creer_datamodule(cat, BATCH_SIZE, IMAGE_SIZE, DATA_DIR)
    afficher_counts_datamodule(dm, cat)

## Cellule 3 — Fonctions utilitaires

In [ ]:
def set_seed(seed):
    """Fixe les graines pour la reproductibilité."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sauvegarder_resultat(ligne_dict, chemin_csv):
    """Ajoute une ligne de résultats dans le CSV selon le schéma fixé."""
    ligne_dict["timestamp"] = datetime.now().isoformat()
    df = pd.DataFrame([ligne_dict])
    if os.path.exists(chemin_csv):
        df.to_csv(chemin_csv, mode="a", header=False, index=False)
    else:
        df.to_csv(chemin_csv, mode="w", header=True, index=False)
    print(f"Résultat sauvegardé : {ligne_dict}")


def sauvegarder_figure(fig, nom):
    """Sauvegarde une figure matplotlib en PNG 150 dpi."""
    chemin = os.path.join(FIGURES_DIR, nom)
    fig.savefig(chemin, dpi=150, bbox_inches="tight")
    print(f"Figure sauvegardée : {chemin}")
    plt.close(fig)

## Cellule 4 — Entraînement et évaluation

In [ ]:
def compter_images_synthetiques(synth_dir, categorie):
    """Compte et affiche le nombre d'images synthétiques par catégorie."""
    chemin = os.path.join(synth_dir, categorie)
    if not os.path.exists(chemin):
        print(f"[!] Pas de dossier synthétique pour {categorie} : {chemin}")
        return 0
    images = [f for f in os.listdir(chemin) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    print(f"Synthétiques {categorie} : {len(images)} images")
    return len(images)


def injecter_synthetiques_dans_train(categorie, synth_dir, data_dir):
    """Copie les images synthétiques dans data/<cat>/train/synthetique/."""
    src = os.path.join(synth_dir, categorie)
    dst = os.path.join(data_dir, categorie, "train", "synthetique")

    if not os.path.exists(src):
        raise FileNotFoundError(f"Dossier synthétique manquant : {src}")

    if os.path.exists(dst):
        for f in os.listdir(dst):
            os.remove(os.path.join(dst, f))
    else:
        os.makedirs(dst, exist_ok=True)

    nb = 0
    for f in os.listdir(src):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))
            nb += 1
    print(f"Injecté {nb} images synthétiques dans {dst}")


def sauvegarder_predictions(predictions, chemin_csv):
    """Sauvegarde les scores et labels du test dans un CSV."""
    lignes = []
    for pred in predictions:
        image_path = pred.get("image_path", [""])[0] if isinstance(pred.get("image_path"), list) else pred.get("image_path", "")
        label = pred.get("label", torch.nan)
        score = pred.get("pred_scores", torch.nan)
        if isinstance(label, torch.Tensor):
            label = label.item()
        if isinstance(score, torch.Tensor):
            score = score.item()
        lignes.append({"image_path": str(image_path), "label": label, "score": score})
    pd.DataFrame(lignes).to_csv(chemin_csv, index=False)
    print(f"Prédictions sauvegardées : {chemin_csv}")


def entrainer_et_evaluer(regime, seed, categorie):
    """Entraîne PatchCore, évalue et sauvegarde les métriques et prédictions."""
    set_seed(seed)
    print(f"\n=== {regime} | {categorie} | seed={seed} ===")

    datamodule = creer_datamodule(
        categorie, BATCH_SIZE, IMAGE_SIZE, DATA_DIR, regime=regime, synth_dir=SYNTH_DIR
    )

    model = Patchcore()
    engine = Engine(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        max_epochs=1,
        enable_progress_bar=True
    )

    engine.fit(model=model, datamodule=datamodule)
    results = engine.test(model=model, datamodule=datamodule)

    auroc_image = float(results[0].get("image_AUROC", float("nan")))
    auroc_pixel = float(results[0].get("pixel_AUROC", float("nan")))
    aupro = float(results[0].get("pixel_AUPRO", float("nan")))

    ligne = {
        "categorie": categorie,
        "regime": regime,
        "seed": seed,
        "auroc_image": auroc_image,
        "auroc_pixel": auroc_pixel,
        "aupro": aupro
    }
    sauvegarder_resultat(ligne, os.path.join(RESULTS_DIR, "results.csv"))

    predictions = engine.predict(model=model, datamodule=datamodule)
    if predictions:
        pred_path = os.path.join(RESULTS_DIR, f"predictions_{regime}_{categorie}_{seed}.csv")
        sauvegarder_predictions(predictions, pred_path)

    return ligne

## Cellule 5 — Entraînement principal (cellule coûteuse, exécutable seule)

In [ ]:
def lire_combinaisons_deja_faites(chemin_csv):
    """Lit les combinaisons (catégorie, régime, graine) déjà effectuées."""
    if not os.path.exists(chemin_csv):
        return set()
    df = pd.read_csv(chemin_csv)
    if df.empty:
        return set()
    return set(zip(df["categorie"], df["regime"], df["seed"].astype(int)))


def executer_experiences():
    """Boucle principale idempotente : seeds × catégories × régimes."""
    chemin_csv = os.path.join(RESULTS_DIR, "results.csv")
    deja_faites = lire_combinaisons_deja_faites(chemin_csv)
    total = len(SEEDS) * len(CATEGORIES) * len(REGIMES)
    print(f"Combinations déjà faites : {len(deja_faites)} / {total}")

    for seed in SEEDS:
        for categorie in CATEGORIES:
            for regime in REGIMES:
                if (categorie, regime, seed) in deja_faites:
                    print(f"Skip {regime} | {categorie} | seed={seed}")
                    continue
                try:
                    entrainer_et_evaluer(regime, seed, categorie)
                except Exception as e:
                    print(f"ERREUR {regime} | {categorie} | seed={seed} : {e}")

executer_experiences()

## Cellule 6 — Agrégation des résultats

In [ ]:
def agreger_resultats(results_csv, summary_csv):
    """Calcule moyenne ± écart-type par régime et par catégorie/régime."""
    df = pd.read_csv(results_csv)
    grouped = df.groupby(["categorie", "regime"])[["auroc_image", "auroc_pixel", "aupro"]]
    summary = grouped.agg(["mean", "std"]).reset_index()
    summary.to_csv(summary_csv, index=False)
    print(summary)
    return summary


summary = agreger_resultats(
    os.path.join(RESULTS_DIR, "results.csv"),
    os.path.join(RESULTS_DIR, "summary.csv")
)

## Cellule 7 — Figures pour le rapport

In [ ]:
def fig1_barplot_auroc_regime(df):
    """Barplot de l'AUROC image moyen par régime avec écart-type."""
    agg = df.groupby("regime")["auroc_image"].agg(["mean", "std"]).reset_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(agg["regime"], agg["mean"], yerr=agg["std"], capsize=5)
    ax.set_ylabel("AUROC image")
    ax.set_title("AUROC image moyen par régime")
    ax.set_ylim([0, 1])
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig1_auroc_par_regime.png")


def fig2_barplot_aupro_regime(df):
    """Barplot de l'AUPRO moyen par régime avec écart-type."""
    agg = df.groupby("regime")["aupro"].agg(["mean", "std"]).reset_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(agg["regime"], agg["mean"], yerr=agg["std"], capsize=5)
    ax.set_ylabel("AUPRO")
    ax.set_title("AUPRO moyen par régime")
    ax.set_ylim([0, 1])
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig2_aupro_par_regime.png")


def fig3_heatmap_auroc(df):
    """Heatmap AUROC (catégories × régimes)."""
    pivot = df.pivot_table(values="auroc_image", index="categorie", columns="regime", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1, ax=ax)
    ax.set_title("AUROC image moyen : catégories × régimes")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig3_heatmap_auroc.png")


def fig4_barplot_auroc_par_graine(df, regime):
    """Barplot des AUROC par graine pour un régime donné."""
    sub = df[df["regime"] == regime]
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=sub, x="categorie", y="auroc_image", hue="seed", ax=ax)
    ax.set_title(f"AUROC image par graine — {regime}")
    ax.set_ylim([0, 1])
    plt.legend(title="Seed")
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig4_auroc_par_graine_{regime}.png")


def fig5_grille_exemples(categorie, synth_dir, data_dir, n_col=5):
    """Grille 3 lignes : normales, défauts réels, défauts synthétiques."""
    chemins_normaux = sorted(Path(data_dir, categorie, "train", "good").glob("*"))[:n_col]
    chemins_defauts = []
    test_dir = Path(data_dir, categorie, "test")
    for d in test_dir.iterdir():
        if d.is_dir() and d.name != "good":
            chemins_defauts.extend(sorted(d.glob("*")))
            if len(chemins_defauts) >= n_col:
                break
    chemins_defauts = chemins_defauts[:n_col]
    chemins_synth = sorted(Path(synth_dir, categorie).glob("*"))[:n_col]

    fig, axes = plt.subplots(3, n_col, figsize=(n_col * 2, 6))
    titres = ["Normaux", "Défauts réels", "Défauts synthétiques"]
    listes = [chemins_normaux, chemins_defauts, chemins_synth]
    for i, (titre, liste) in enumerate(zip(titres, listes)):
        axes[i, 0].set_ylabel(titre)
        for j in range(n_col):
            ax = axes[i, j]
            ax.axis("off")
            if j < len(liste):
                img = Image.open(liste[j]).convert("RGB")
                ax.imshow(img)
    plt.suptitle(f"Exemples — {categorie}")
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig5_exemples_{categorie}.png")


def fig6_analyse_erreurs(predictions_csv, n_exemples=10):
    """Affiche les exemples de test les plus mal classés."""
    df = pd.read_csv(predictions_csv)
    df["erreur"] = np.abs(df["label"] - df["score"])
    pires = df.sort_values("erreur", ascending=False).head(n_exemples)
    n_col = min(n_exemples, 5)
    n_row = int(np.ceil(n_exemples / n_col))
    fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * 2, n_row * 2))
    axes = np.array(axes).flatten()
    for idx, (_, row) in enumerate(pires.iterrows()):
        if idx >= len(axes):
            break
        img = Image.open(row["image_path"]).convert("RGB")
        axes[idx].imshow(img)
        axes[idx].set_title(f"score={row['score']:.2f}\nlabel={int(row['label'])}")
        axes[idx].axis("off")
    for idx in range(len(pires), len(axes)):
        axes[idx].axis("off")
    plt.suptitle("Exemples les plus mal classés")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig6_analyse_erreurs.png")


results_df = pd.read_csv(os.path.join(RESULTS_DIR, "results.csv"))
fig1_barplot_auroc_regime(results_df)
fig2_barplot_aupro_regime(results_df)
fig3_heatmap_auroc(results_df)
for regime in REGIMES:
    fig4_barplot_auroc_par_graine(results_df, regime)
for categorie in CATEGORIES:
    fig5_grille_exemples(categorie, SYNTH_DIR, DATA_DIR)
pred_csv = os.path.join(RESULTS_DIR, f"predictions_R1_reel_seul_{CATEGORIES[0]}_{SEEDS[0]}.csv")
if os.path.exists(pred_csv):
    fig6_analyse_erreurs(pred_csv)

## Cellule 8 — Analyse de quasi-duplication

In [ ]:
def charger_modele_features():
    """Charge ResNet50 ImageNet sans la dernière couche, gelé."""
    model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
    model = torch.nn.Sequential(*list(model.children())[:-1])
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model.to(DEVICE)


def extraire_features_images(liste_chemins, model, device, batch_size=32):
    """Extrait les features ResNet50 pour une liste d'images."""
    preprocess = torchvision.transforms.Compose([
        torchvision.transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    features = []
    for i in tqdm(range(0, len(liste_chemins), batch_size), desc="Extraction features"):
        batch_paths = liste_chemins[i:i + batch_size]
        tensors = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB")
            tensors.append(preprocess(img))
        batch = torch.stack(tensors).to(device)
        with torch.no_grad():
            out = model(batch).squeeze()
        features.append(out.cpu().numpy())
    return np.vstack(features)


def lister_defauts_reels_test(data_dir, categorie):
    """Retourne les chemins des vrais défauts du split test (galerie de référence)."""
    test_dir = Path(data_dir, categorie, "test")
    chemins = []
    for d in test_dir.iterdir():
        if d.is_dir() and d.name != "good":
            chemins.extend([p for p in d.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])
    print(f"Défauts réels test {categorie} : {len(chemins)} images")
    return chemins


def lister_images_synthetiques(synth_dir, categorie):
    """Retourne les chemins des images synthétiques."""
    chemin = Path(synth_dir, categorie)
    if not chemin.exists():
        return []
    chemins = [p for p in chemin.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
    return chemins


def analyser_quasi_duplication(features_defauts, features_synth, seuils):
    """Calcule distances cosinus au plus proche vrai défaut et taux sous seuils."""
    nn = NearestNeighbors(n_neighbors=1, metric="cosine")
    nn.fit(features_defauts)
    distances, indices = nn.kneighbors(features_synth)
    distances = distances.flatten()
    indices = indices.flatten()
    taux = {s: float(np.mean(distances < s)) for s in seuils}
    return distances, indices, taux


def fig7_histogramme_distances(distances, categorie):
    """Histogramme des distances au plus proche vrai défaut."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(distances, bins=50, edgecolor="black")
    ax.set_xlabel("Distance cosinus au plus proche défaut réel")
    ax.set_ylabel("Nombre d'images synthétiques")
    ax.set_title(f"Quasi-duplication — {categorie}")
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig7_histogramme_distances_{categorie}.png")


def fig8_taux_quasi_duplication(taux_dict, categorie):
    """Barplot des taux d'images synthétiques sous chaque seuil."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar([str(s) for s in taux_dict.keys()], [v * 100 for v in taux_dict.values()])
    ax.set_xlabel("Seuil de distance cosinus")
    ax.set_ylabel("% d'images synthétiques")
    ax.set_title(f"Taux de quasi-duplication — {categorie}")
    ax.set_ylim([0, 100])
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig8_taux_quasi_duplication_{categorie}.png")


def fig9_exemples_voisins(chemins_synth, chemins_defauts, indices, distances, categorie, n=10):
    """Affiche côte à côte synthétiques et leur plus proche voisin réel."""
    n = min(n, len(chemins_synth))
    fig, axes = plt.subplots(n, 2, figsize=(4, n * 2))
    for i in range(n):
        img_synth = Image.open(chemins_synth[i]).convert("RGB")
        img_voisin = Image.open(chemins_defauts[indices[i]]).convert("RGB")
        axes[i, 0].imshow(img_synth)
        axes[i, 0].set_title("Synthétique")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(img_voisin)
        axes[i, 1].set_title(f"Vrai défaut (dist={distances[i]:.3f})")
        axes[i, 1].axis("off")
    plt.suptitle(f"Voisinage — {categorie}")
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig9_exemples_voisins_{categorie}.png")


model_features = charger_modele_features()
seuils = [0.05, 0.10, 0.15]

for categorie in CATEGORIES:
    chemins_defauts = lister_defauts_reels_test(DATA_DIR, categorie)
    chemins_synth = lister_images_synthetiques(SYNTH_DIR, categorie)
    if not chemins_synth or not chemins_defauts:
        print(f"Skip {categorie} : pas assez d'images")
        continue

    feats_defauts = extraire_features_images(chemins_defauts, model_features, DEVICE)
    feats_synth = extraire_features_images(chemins_synth, model_features, DEVICE)
    distances, indices, taux = analyser_quasi_duplication(feats_defauts, feats_synth, seuils)

    distances_csv = os.path.join(RESULTS_DIR, f"distances_quasi_duplication_{categorie}.csv")
    pd.DataFrame({
        "synthetique": [str(p) for p in chemins_synth],
        "voisin_reel": [str(chemins_defauts[i]) for i in indices],
        "distance": distances
    }).to_csv(distances_csv, index=False)
    print(f"Distances sauvegardées : {distances_csv}")
    print(f"Taux sous seuils : {taux}")

    fig7_histogramme_distances(distances, categorie)
    fig8_taux_quasi_duplication(taux, categorie)
    fig9_exemples_voisins(chemins_synth, chemins_defauts, indices, distances, categorie)

## Cellule 9 — Ablations

In [ ]:
def executer_ablations_aug(categorie, seed):
    """Teste R2 en retirant successivement flip, rotation, bruit."""
    set_seed(seed)
    configs = {
        "flip_seul": [A.HorizontalFlip(p=0.5)],
        "rotation_seule": [A.Rotate(limit=15, p=0.5)],
        "bruit_seul": [A.GaussNoise(var_limit=(10.0, 50.0), p=0.3)]
    }
    base = [A.Resize(IMAGE_SIZE, IMAGE_SIZE), A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()]
    lignes = []
    for nom, transforms in configs.items():
        print(f"\n--- Ablations R2 : {nom} ---")
        train_transform = A.Compose(transforms + base)
        datamodule = Mvtec(
            root=DATA_DIR, category=categorie, image_size=(IMAGE_SIZE, IMAGE_SIZE),
            train_batch_size=BATCH_SIZE, eval_batch_size=BATCH_SIZE,
            num_workers=0, task="SEGMENTATION",
            train_transform=train_transform
        )
        datamodule.prepare_data()
        datamodule.setup()
        model = Patchcore()
        engine = Engine(accelerator="gpu" if torch.cuda.is_available() else "cpu", devices=1, max_epochs=1)
        engine.fit(model=model, datamodule=datamodule)
        results = engine.test(model=model, datamodule=datamodule)
        lignes.append({
            "categorie": categorie, "seed": seed, "ablation": nom,
            "auroc_image": float(results[0].get("image_AUROC", float("nan"))),
            "auroc_pixel": float(results[0].get("pixel_AUROC", float("nan"))),
            "aupro": float(results[0].get("pixel_AUPRO", float("nan")))
        })
    df = pd.DataFrame(lignes)
    df.to_csv(os.path.join(RESULTS_DIR, "ablations_aug.csv"), index=False)
    return df


def filtrer_synthetiques_quasi_dupliques(categorie, synth_dir, data_dir, results_dir):
    """Retire les synthétiques sous la médiane des distances au vrai défaut."""
    distances_csv = os.path.join(results_dir, f"distances_quasi_duplication_{categorie}.csv")
    df = pd.read_csv(distances_csv)
    seuil = df["distance"].median()
    a_garder = df[df["distance"] >= seuil]["synthetique"].tolist()
    print(f"Filtrage {categorie} : garde {len(a_garder)} / {len(df)} (seuil médiane={seuil:.4f})")
    return a_garder, seuil


def executer_ablation_synth(categorie, seed):
    """Entraîne R3 sans les synthétiques quasi-dupliqués."""
    set_seed(seed)
    print(f"\n--- Ablations R3 : filtrage quasi-dupliqués | {categorie} | seed={seed} ---")

    a_garder, seuil = filtrer_synthetiques_quasi_dupliques(categorie, SYNTH_DIR, DATA_DIR, RESULTS_DIR)
    src = Path(SYNTH_DIR, categorie)
    dst = Path(DATA_DIR, categorie, "train", "synthetique_filtrée")
    if dst.exists():
        for f in dst.iterdir():
            f.unlink()
    dst.mkdir(parents=True, exist_ok=True)
    for p in a_garder:
        shutil.copy(p, dst / Path(p).name)

    datamodule = Mvtec(
        root=DATA_DIR, category=categorie, image_size=(IMAGE_SIZE, IMAGE_SIZE),
        train_batch_size=BATCH_SIZE, eval_batch_size=BATCH_SIZE,
        num_workers=0, task="SEGMENTATION"
    )
    datamodule.prepare_data()
    datamodule.setup()
    model = Patchcore()
    engine = Engine(accelerator="gpu" if torch.cuda.is_available() else "cpu", devices=1, max_epochs=1)
    engine.fit(model=model, datamodule=datamodule)
    results = engine.test(model=model, datamodule=datamodule)
    ligne = {
        "categorie": categorie, "seed": seed, "ablation": "synth_filtree", "seuil": seuil,
        "auroc_image": float(results[0].get("image_AUROC", float("nan"))),
        "auroc_pixel": float(results[0].get("pixel_AUROC", float("nan"))),
        "aupro": float(results[0].get("pixel_AUPRO", float("nan")))
    }
    df = pd.DataFrame([ligne])
    df.to_csv(os.path.join(RESULTS_DIR, "ablations_synth.csv"), index=False)
    return df


def fig10_barplot_ablations(df_aug, df_synth):
    """Barplot comparant les ablations data-aug et filtrage synthétique."""
    df_aug = df_aug.copy()
    df_aug["type"] = "aug"
    df_synth = df_synth.copy()
    df_synth["type"] = "synth"
    df = pd.concat([df_aug[["ablation", "auroc_image", "type"]], df_synth[["ablation", "auroc_image", "type"]]])
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=df, x="ablation", y="auroc_image", hue="type", ax=ax)
    ax.set_ylim([0, 1])
    ax.set_title("Ablations : impact sur l'AUROC image")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig10_ablations.png")


cat_ref = CATEGORIES[0]
seed_ref = SEEDS[0]
df_aug = executer_ablations_aug(cat_ref, seed_ref)
df_synth = executer_ablation_synth(cat_ref, seed_ref)
fig10_barplot_ablations(df_aug, df_synth)

## Cellule 10 — Rapport automatique

In [ ]:
def generer_rapport(results_csv, summary_csv, ablations_aug_csv, ablations_synth_csv, figures_dir, output_md):
    """Génère README_results.md avec tableaux et liste des figures."""
    results = pd.read_csv(results_csv)
    summary = pd.read_csv(summary_csv)
    abl_aug = pd.read_csv(ablations_aug_csv) if os.path.exists(ablations_aug_csv) else None
    abl_synth = pd.read_csv(ablations_synth_csv) if os.path.exists(ablations_synth_csv) else None
    figures = sorted(Path(figures_dir).glob("*.png"))

    lignes = ["# Rapport de résultats — IND-3\n"]
    lignes.append("## Résumé des métriques par régime\n")
    lignes.append(summary.to_markdown(index=False))
    lignes.append("\n\n")

    lignes.append("## Résultats bruts\n")
    lignes.append(results.to_markdown(index=False))
    lignes.append("\n\n")

    if abl_aug is not None:
        lignes.append("## Ablations data-aug (R2)\n")
        lignes.append(abl_aug.to_markdown(index=False))
        lignes.append("\n\n")
    if abl_synth is not None:
        lignes.append("## Ablations synthétiques (R3 filtré)\n")
        lignes.append(abl_synth.to_markdown(index=False))
        lignes.append("\n\n")

    lignes.append("## Note méthodologique\n")
    lignes.append("> Les images de défauts réels utilisées dans l'analyse de quasi-duplication proviennent **uniquement du split test** de MVTec AD. \n")
    lignes.append("> Elles servent de **galerie de référence** pour le calcul de distance (comme le FID) et ne sont **jamais utilisées** pour l'entraînement ni la sélection de modèle. \n")
    lignes.append("\n")

    lignes.append("## Figures générées\n")
    for fig in figures:
        lignes.append(f"- {fig.name}")

    with open(output_md, "w") as f:
        f.write("\n".join(lignes))
    print(f"Rapport généré : {output_md}")


generer_rapport(
    os.path.join(RESULTS_DIR, "results.csv"),
    os.path.join(RESULTS_DIR, "summary.csv"),
    os.path.join(RESULTS_DIR, "ablations_aug.csv"),
    os.path.join(RESULTS_DIR, "ablations_synth.csv"),
    FIGURES_DIR,
    os.path.join(RESULTS_DIR, "README_results.md")
)